In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import AclPermission

# setup_once/03_refresh_setup (5.11): run ONCE by Management (admin), from any Git folder. Rerun-safe. Prints no ids.
#  - secret scope lh_refresh: the refresh workflow's deploy account (group lh_deploy) writes the key, the job accounts
#    of the source environments (prod, test) read it; no person reads it
#  - each source job account may write ONLY into the target's ops.refresh_in volume (prod -> test, test -> dev);
#    nothing gives a lower environment any access to a higher one
dbutils.widgets.dropdown("mode", "plan", ["plan", "apply"])
apply = dbutils.widgets.get("mode") == "apply"
w = WorkspaceClient()
sp = {s.display_name: s.application_id for s in w.service_principals.list()}
FLOWS = [("prod", "test"), ("test", "dev")]
readers = [f"sp_{c}_{src}" for c in ("bhg", "fqf") for src, _ in FLOWS]
missing = [r for r in readers if r not in sp]
if missing:
    raise ValueError(f"job accounts not found: {missing}")

if "lh_refresh" not in {s.name for s in w.secrets.list_scopes()}:
    print("scope lh_refresh: create")
    if apply:
        w.secrets.create_scope(scope="lh_refresh")
else:
    print("scope lh_refresh: already there")
acl = [("lh_deploy", AclPermission.WRITE)] + [(sp[r], AclPermission.READ) for r in readers]
for principal, perm in acl:
    print(f"lh_refresh: {principal if principal == 'lh_deploy' else [k for k, v in sp.items() if v == principal][0]} -> {perm.value}")
    if apply:
        w.secrets.put_acl(scope="lh_refresh", principal=principal, permission=perm)

for c in ("bhg", "fqf"):
    for src, tgt in FLOWS:
        who, cat = sp[f"sp_{c}_{src}"], f"{c}_{tgt}"
        stmts = [f"GRANT USE CATALOG ON CATALOG {cat} TO `{who}`",
                 f"GRANT USE SCHEMA ON SCHEMA {cat}.ops TO `{who}`",
                 f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {cat}.ops.refresh_in TO `{who}`"]
        print(f"sp_{c}_{src} -> {cat}.ops.refresh_in: write")
        if apply:
            for q in stmts:
                spark.sql(q)
print("mode", "apply: done" if apply else "plan: nothing changed")